In [1]:
import torch

In [2]:
print(torch.__version__)

2.9.1+cpu


In [3]:
!nvidia-smi

Tue Nov 25 02:08:51 2025       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 581.57                 Driver Version: 581.57         CUDA Version: 13.0     |
+-----------------------------------------+------------------------+----------------------+
| GPU  Name                  Driver-Model | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  NVIDIA GeForce RTX 3060      WDDM  |   00000000:07:00.0  On |                  N/A |
| 31%   46C    P0             41W /  170W |    1392MiB /  12288MiB |      7%      Default |
|                                         |                        |                  N/A |
+-----------------------------------------+-----

# Load video dataset and the XML bounding boxes. 

In [4]:
import os
import cv2
import glob
import torch
import numpy as np
import xml.etree.ElementTree as ET
import matplotlib.pyplot as plt

# ---- CONFIGURATION ----
ROOT_DIR = "F:/IDDPedestrian"

# Define the 9 sets
DATA_SETS = [f"gp_set_{str(i).zfill(4)}" for i in range(1, 10)] 

print(f"Root Directory: {ROOT_DIR}")
print(f"Processing Sets: {DATA_SETS}")

# Check for GPU
device = 'cuda:0' if torch.cuda.is_available() else 'cpu'
print(f"Inference Device: {device}")

Root Directory: F:/IDDPedestrian
Processing Sets: ['gp_set_0001', 'gp_set_0002', 'gp_set_0003', 'gp_set_0004', 'gp_set_0005', 'gp_set_0006', 'gp_set_0007', 'gp_set_0008', 'gp_set_0009']
Inference Device: cpu


In [5]:
import os
import glob
import xml.etree.ElementTree as ET
import cv2
import numpy as np
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, f1_score, roc_auc_score
from tqdm import tqdm
import pickle

# ---- CONFIGURATION ----
ROOT_DIR = "F:/IDDPedestrian"
DATA_SETS = [f"gp_set_{str(i).zfill(4)}" for i in range(1, 10)]

# Paper Parameters (Section IV. A. Experimental Settings)
# Observation period: 0.5s. At ~30FPS, this is roughly 15 frames.
OBSERVATION_LEN = 15 
PREDICTION_HORIZON = 45 # 1.5s into the future (for Trajectory)

# Where to save the mathematical dataset for the AI
PROCESSED_DATA_PATH = os.path.join(ROOT_DIR, "idd_ped_training_data.pkl")

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Running on device: {device}")
print(f"Processing sets: {DATA_SETS}")

Running on device: cpu
Processing sets: ['gp_set_0001', 'gp_set_0002', 'gp_set_0003', 'gp_set_0004', 'gp_set_0005', 'gp_set_0006', 'gp_set_0007', 'gp_set_0008', 'gp_set_0009']


In [6]:
def parse_cvat_xml_tracks(xml_file, video_width, video_height):
    """
    Parses CVAT XML to extract full tracks with attributes.
    Returns: Dict { track_id: { 'frames': [], 'bbox': [], 'attributes': [] } }
    """
    if xml_file is None or not os.path.exists(xml_file):
        return {}

    try:
        tree = ET.parse(xml_file)
        root = tree.getroot()
        tracks_data = {}
        
        for track in root.findall('track'):
            track_id = int(track.get('id'))
            label = track.get('label')
            
            # Filter: The paper focuses on Pedestrians
            if 'pedestrian' not in label.lower() and 'rider' not in label.lower():
                continue

            tracks_data[track_id] = {
                'label': label,
                'frames': [],
                'bboxes': [],     # Normalized [x1, y1, x2, y2]
                'behavior': [],   # Store behavior tags to determine intention
                'occluded': []
            }
            
            for box in track.findall('box'):
                frame_num = int(box.get('frame'))
                xtl = float(box.get('xtl'))
                ytl = float(box.get('ytl'))
                xbr = float(box.get('xbr'))
                ybr = float(box.get('ybr'))
                occluded = int(box.get('occluded', 0))
                
                # Extract Attributes for Ground Truth (Crossing vs Not Crossing)
                # We look for attributes named 'action', 'behavior', or 'cross'
                behavior_tags = []
                for attr in box.findall('attribute'):
                    attr_name = attr.get('name').lower()
                    attr_val = attr.text.lower()
                    behavior_tags.append(attr_val)

                # Normalize coordinates (0-1 range) for Neural Networks efficiency
                norm_box = [xtl/video_width, ytl/video_height, xbr/video_width, ybr/video_height]

                tracks_data[track_id]['frames'].append(frame_num)
                tracks_data[track_id]['bboxes'].append(norm_box)
                tracks_data[track_id]['behavior'].append(behavior_tags)
                tracks_data[track_id]['occluded'].append(occluded)

        return tracks_data
    except Exception as e:
        print(f"Error parsing {xml_file}: {e}")
        return {}

In [7]:
def extract_dataset(sets_to_process):
    all_samples = [] 
    
    for set_name in sets_to_process:
        # Define paths
        video_dir = os.path.join(ROOT_DIR, "videos", "gopro", set_name)
        ped_dir = os.path.join(ROOT_DIR, "annotations", "gopro", "annotations", "gopro", set_name)
        
        if not os.path.exists(video_dir): continue

        # Get video list
        vid_files = sorted(glob.glob(os.path.join(video_dir, "*.mp4")))
        
        for vid_path in tqdm(vid_files, desc=f"Processing {set_name}"):
            base_name = os.path.basename(vid_path).split('.')[0]
            
            # Find matching XML
            xml_path = None
            if os.path.exists(ped_dir):
                # Simple matching logic
                candidates = glob.glob(os.path.join(ped_dir, f"*{base_name}*.xml"))
                if candidates: xml_path = candidates[0]
            
            if not xml_path: continue

            # Get Video Dimensions for normalization
            cap = cv2.VideoCapture(vid_path)
            w = cap.get(cv2.CAP_PROP_FRAME_WIDTH)
            h = cap.get(cv2.CAP_PROP_FRAME_HEIGHT)
            cap.release()
            
            if w == 0 or h == 0: continue

            # Parse Tracks
            tracks = parse_cvat_xml_tracks(xml_path, w, h)
            
            # Slice Tracks into Sequences
            for t_id, t_data in tracks.items():
                frames = t_data['frames']
                bboxes = t_data['bboxes']
                behaviors = t_data['behavior']
                
                # Need at least Observation length
                if len(frames) < OBSERVATION_LEN:
                    continue
                
                # Sliding window to create samples
                # We take a window of 15 frames as input
                for i in range(len(frames) - OBSERVATION_LEN):
                    seq_bboxes = bboxes[i : i+OBSERVATION_LEN]
                    seq_behaviors = behaviors[i : i+OBSERVATION_LEN]
                    
                    # --- DETERMINE LABEL (Crossing vs Not Crossing) ---
                    # Logic: If any frame in the future or current sequence indicates crossing
                    is_crossing = 0
                    flat_behaviors = [item for sublist in seq_behaviors for item in sublist]
                    
                    # Keywords based on IDD-PeD paper descriptions
                    crossing_keywords = ['cross', 'jaywalk', 'crossing', 'jaywalking']
                    
                    if any(key in b for b in flat_behaviors for key in crossing_keywords):
                        is_crossing = 1
                    
                    # --- CREATE FEATURES ---
                    # Feature vector: [x1, y1, x2, y2, width, height]
                    seq_features = []
                    for b in seq_bboxes:
                        bw = b[2] - b[0]
                        bh = b[3] - b[1]
                        seq_features.append(b + [bw, bh])
                        
                    all_samples.append({
                        'features': np.array(seq_features, dtype=np.float32),
                        'label': is_crossing,
                        'video': base_name,
                        'track_id': t_id
                    })
                    
    return all_samples

# ---- EXECUTE EXTRACTION ----
if not os.path.exists(PROCESSED_DATA_PATH):
    print("Extracting dataset from raw XMLs (this may take time)...")
    dataset_samples = extract_dataset(DATA_SETS)
    print(f"Extracted {len(dataset_samples)} samples.")
    with open(PROCESSED_DATA_PATH, 'wb') as f:
        pickle.dump(dataset_samples, f)
else:
    print("Loading cached dataset...")
    with open(PROCESSED_DATA_PATH, 'rb') as f:
        dataset_samples = pickle.load(f)

if len(dataset_samples) > 0:
    print(f"Sample 0 Feature Shape: {dataset_samples[0]['features'].shape}") # Should be (15, 6)
    print(f"Sample 0 Label: {dataset_samples[0]['label']}")

Extracting dataset from raw XMLs (this may take time)...


Processing gp_set_0001:  88%|████████████████████████████████████████████████▏      | 7/8 [00:13<00:01,  1.84s/it]

Error parsing F:/IDDPedestrian\annotations\gopro\annotations\gopro\gp_set_0001\gp_set_0001_vid_0008.xml: 'NoneType' object has no attribute 'lower'


Processing gp_set_0002:  25%|█████████████▊                                         | 1/4 [00:05<00:16,  5.35s/it]

Error parsing F:/IDDPedestrian\annotations\gopro\annotations\gopro\gp_set_0002\gp_set_0002_vid_0002.xml: 'NoneType' object has no attribute 'lower'


...

Error parsing F:/IDDPedestrian\annotations\gopro\annotations\gopro\gp_set_0006\gp_set_0006_vid_0001.xml: 'NoneType' object has no attribute 'lower'


Processing gp_set_0006:  67%|████████████████████████████████████▋                  | 2/3 [00:03<00:01,  1.33s/it]

Error parsing F:/IDDPedestrian\annotations\gopro\annotations\gopro\gp_set_0006\gp_set_0006_vid_0002.xml: 'NoneType' object has no attribute 'lower'


Processing gp_set_0009: 100%|███████████████████████████████████████████████████████| 2/2 [00:01<00:00,  1.61it/s]


Extracted 272304 samples.
Sample 0 Feature Shape: (15, 6)
Sample 0 Label: 0


In [8]:
class IDDPedDataset(Dataset):
    def __init__(self, samples):
        self.samples = samples

    def __len__(self):
        return len(self.samples)

    def __getitem__(self, idx):
        sample = self.samples[idx]
        # Input: Sequence of BBoxes (15, 6)
        features = torch.tensor(sample['features'], dtype=torch.float32)
        # Output: Class (0 or 1) -> unsqueeze to make it shape (1)
        label = torch.tensor(sample['label'], dtype=torch.float32).unsqueeze(0)
        
        return features, label

# Split Data (70% Train, 30% Test as per Paper Section III.C)
# We perform stratified split to ensure crossing/not-crossing ratio is balanced
labels_list = [s['label'] for s in dataset_samples]
train_data, test_data = train_test_split(dataset_samples, test_size=0.3, random_state=42, stratify=labels_list)

train_dataset = IDDPedDataset(train_data)
test_dataset = IDDPedDataset(test_data)

train_loader = DataLoader(train_dataset, batch_size=64, shuffle=True)
test_loader = DataLoader(test_dataset, batch_size=64, shuffle=False)

print(f"Train samples: {len(train_dataset)}")
print(f"Test samples: {len(test_dataset)}")

Train samples: 190612
Test samples: 81692


In [9]:
class PedestrianIntentRNN(nn.Module):
    def __init__(self, input_size=6, hidden_size=128, num_layers=1):
        super(PedestrianIntentRNN, self).__init__()
        
        # LSTM extracts temporal features from the sequence of boxes
        self.lstm = nn.LSTM(
            input_size=input_size,
            hidden_size=hidden_size,
            num_layers=num_layers,
            batch_first=True,
            dropout=0.2 if num_layers > 1 else 0
        )
        
        # Fully Connected layers to predict binary outcome (Cross / No Cross)
        self.fc = nn.Sequential(
            nn.Linear(hidden_size, 64),
            nn.ReLU(),
            nn.Linear(64, 1), # Output size 1
            nn.Sigmoid()      # Squash to 0-1 probability
        )

    def forward(self, x):
        # x shape: [batch, seq_len, input_size]
        
        # Run LSTM
        # out shape: [batch, seq_len, hidden_size]
        out, (hn, cn) = self.lstm(x)
        
        # We only care about the output of the LAST frame in the sequence
        last_hidden_state = out[:, -1, :]
        
        # Predict
        prediction = self.fc(last_hidden_state)
        return prediction

# Initialize Model
model = PedestrianIntentRNN().to(device)
criterion = nn.BCELoss() # Binary Cross Entropy for 0/1 classification
optimizer = optim.Adam(model.parameters(), lr=0.001)

print(model)

PedestrianIntentRNN(
  (lstm): LSTM(6, 128, batch_first=True)
  (fc): Sequential(
    (0): Linear(in_features=128, out_features=64, bias=True)
    (1): ReLU()
    (2): Linear(in_features=64, out_features=1, bias=True)
    (3): Sigmoid()
  )
)


In [11]:
# ---- FIX CONFIGURATION ----
# Ensure we have a place to save the models
if 'ROOT_DIR' not in globals():
    ROOT_DIR = "F:/IDDPedestrian"

OUTPUT_ROOT = os.path.join(ROOT_DIR, "saved_models")
if not os.path.exists(OUTPUT_ROOT):
    os.makedirs(OUTPUT_ROOT)

# ---- TRAINING PARAMETERS ----
EPOCHS = 10

def train_epoch(loader, model, opt, crit):
    model.train()
    total_loss = 0
    
    loop = tqdm(loader, desc="Training", leave=False)
    for features, labels in loop:
        features, labels = features.to(device), labels.to(device)
        
        opt.zero_grad()
        outputs = model(features)
        loss = crit(outputs, labels)
        
        loss.backward()
        opt.step()
        
        total_loss += loss.item()
        loop.set_postfix(loss=loss.item())
        
    return total_loss / len(loader)

def evaluate(loader, model):
    model.eval()
    all_preds = []
    all_labels = []
    
    with torch.no_grad():
        for features, labels in loader:
            features, labels = features.to(device), labels.to(device)
            outputs = model(features)
            
            all_preds.extend(outputs.cpu().numpy())
            all_labels.extend(labels.cpu().numpy())
            
    # Convert list to numpy
    all_preds = np.array(all_preds)
    all_labels = np.array(all_labels)
    
    # Binarize predictions for Accuracy/F1 (Threshold 0.5)
    bin_preds = (all_preds > 0.5).astype(int)
    
    acc = accuracy_score(all_labels, bin_preds)
    f1 = f1_score(all_labels, bin_preds)
    try:
        auc = roc_auc_score(all_labels, all_preds)
    except:
        auc = 0.5 # Fallback if only one class exists
        
    return acc, f1, auc

# ---- START TRAINING ----
print(f"{'='*10} STARTING TRAINING {'='*10}")
print(f"Saving models to: {OUTPUT_ROOT}")
best_auc = 0.0

for epoch in range(EPOCHS):
    train_loss = train_epoch(train_loader, model, optimizer, criterion)
    val_acc, val_f1, val_auc = evaluate(test_loader, model)
    
    print(f"Epoch [{epoch+1}/{EPOCHS}] Loss: {train_loss:.4f} | Acc: {val_acc:.4f} | F1: {val_f1:.4f} | AUC: {val_auc:.4f}")
    
    # Save the best model
    if val_auc > best_auc:
        best_auc = val_auc
        save_path = os.path.join(OUTPUT_ROOT, "idd_ped_baseline_best.pth")
        torch.save(model.state_dict(), save_path)
        print(f"  --> New Best Model Saved!")

print(f"\nTraining Complete. Check results in: {OUTPUT_ROOT}")

========== STARTING TRAINING ==========
Saving models to: F:/IDDPedestrian\saved_models


Epoch [1/10] Loss: 0.0096 | Acc: 0.9988 | F1: 0.0000 | AUC: 0.7946
  --> New Best Model Saved!


Epoch [2/10] Loss: 0.0095 | Acc: 0.9988 | F1: 0.0000 | AUC: 0.7813


Epoch [3/10] Loss: 0.0094 | Acc: 0.9988 | F1: 0.0000 | AUC: 0.7811


Epoch [4/10] Loss: 0.0093 | Acc: 0.9988 | F1: 0.0000 | AUC: 0.7844


...

Epoch [6/10] Loss: 0.0089 | Acc: 0.9988 | F1: 0.0000 | AUC: 0.7954
  --> New Best Model Saved!


Epoch [7/10] Loss: 0.0088 | Acc: 0.9988 | F1: 0.0000 | AUC: 0.8002
  --> New Best Model Saved!


Epoch [8/10] Loss: 0.0087 | Acc: 0.9988 | F1: 0.0000 | AUC: 0.8037
  --> New Best Model Saved!


Epoch [9/10] Loss: 0.0087 | Acc: 0.9988 | F1: 0.0000 | AUC: 0.8096
  --> New Best Model Saved!


Epoch [10/10] Loss: 0.0086 | Acc: 0.9988 | F1: 0.0000 | AUC: 0.8137
  --> New Best Model Saved!

Training Complete. Check results in: F:/IDDPedestrian\saved_models


In [12]:
import os
import glob
import xml.etree.ElementTree as ET
import cv2
import numpy as np
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader
from sklearn.model_selection import train_test_split
from sklearn.metrics import roc_auc_score, mean_squared_error
from tqdm import tqdm
import pickle

# ---- CONFIGURATION ----
if 'ROOT_DIR' not in globals():
    ROOT_DIR = "F:/IDDPedestrian"

OUTPUT_ROOT = os.path.join(ROOT_DIR, "advanced_results")
if not os.path.exists(OUTPUT_ROOT): os.makedirs(OUTPUT_ROOT)

DATA_SETS = [f"gp_set_{str(i).zfill(4)}" for i in range(1, 10)]

# Paper Settings
OBS_LEN = 15      # 0.5s input
PRED_LEN = 45     # 1.5s output (for Table V Trajectory)
IMAGE_W, IMAGE_H = 1920, 1080 # Standard GoPro dims, used for normalization

# Save path
ADVANCED_DATA_PATH = os.path.join(ROOT_DIR, "idd_ped_advanced_data.pkl")

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Device: {device} | Output: {OUTPUT_ROOT}")

Device: cpu | Output: F:/IDDPedestrian\advanced_results


In [13]:
def parse_xml_advanced(xml_file):
    """
    Extracts tracks PLUS metadata: Occlusion, Interaction, Signalized status.
    """
    if xml_file is None or not os.path.exists(xml_file): return {}

    try:
        tree = ET.parse(xml_file)
        root = tree.getroot()
        tracks_data = {}
        
        # 1. Check Scene Context (Signalized?)
        # We assume if a traffic light or crosswalk exists in the XML, the scene is signalized.
        has_signal = False
        has_crosswalk = False
        for track in root.findall('track'):
            lbl = track.get('label').lower()
            if 'traffic_light' in lbl or 'signal' in lbl: has_signal = True
            if 'crosswalk' in lbl: has_crosswalk = True
        
        is_signalized = has_signal or has_crosswalk

        # 2. Parse Pedestrian Tracks
        for track in root.findall('track'):
            track_id = int(track.get('id'))
            label = track.get('label')
            
            if 'pedestrian' not in label.lower(): continue

            # Extract Track-level attributes (sometimes interaction is here)
            track_interaction = False
            
            frames = []
            bboxes = []
            frame_attrs = [] # Stores (occluded, interaction) per frame

            for box in track.findall('box'):
                frame_num = int(box.get('frame'))
                xtl, ytl = float(box.get('xtl')), float(box.get('ytl'))
                xbr, ybr = float(box.get('xbr')), float(box.get('ybr'))
                
                # --- Attribute Parsing ---
                occluded = int(box.get('occluded', 0))
                
                # Check for interaction attribute
                interaction = 0
                for attr in box.findall('attribute'):
                    name = attr.get('name').lower()
                    val = attr.text.lower()
                    if 'interaction' in name and 'true' in val:
                        interaction = 1
                    # Also check for Crossing label here
                    if 'action' in name:
                        pass # handled in extraction logic

                # Normalize Boxes
                norm_box = [xtl/IMAGE_W, ytl/IMAGE_H, xbr/IMAGE_W, ybr/IMAGE_H]
                
                frames.append(frame_num)
                bboxes.append(norm_box)
                frame_attrs.append({'occ': occluded, 'int': interaction})

            tracks_data[track_id] = {
                'frames': frames,
                'bboxes': bboxes,
                'attrs': frame_attrs,
                'signalized': 1 if is_signalized else 0
            }
            
        return tracks_data
    except Exception as e:
        return {}

In [14]:
def process_dataset_advanced(sets):
    data = []
    
    for set_name in sets:
        # Hueristic for Illumination based on set ID (Adjust based on your actual data visual check)
        # Assuming sets 7-9 are night for demonstration, or relying on tags if available.
        # Ideally, you parse this from a "time_of_day" tag in XML.
        is_night = 1 if 'night' in set_name or int(set_name.split('_')[-1]) >= 8 else 0
        
        vid_dir = os.path.join(ROOT_DIR, "videos", "gopro", set_name)
        xml_dir = os.path.join(ROOT_DIR, "annotations", "gopro", "annotations", "gopro", set_name)
        
        if not os.path.exists(vid_dir): continue
        
        xml_files = glob.glob(os.path.join(xml_dir, "*.xml"))
        
        for xml_path in tqdm(xml_files, desc=f"Set {set_name}"):
            tracks = parse_xml_advanced(xml_path)
            
            for tid, tdata in tracks.items():
                frames = tdata['frames']
                bboxes = tdata['bboxes']
                attrs = tdata['attrs']
                is_sig = tdata['signalized']
                
                # We need Input(15) + Future(45) = 60 frames for PTP evaluation
                total_len = OBS_LEN + PRED_LEN
                
                if len(frames) < total_len: continue
                
                for i in range(len(frames) - total_len):
                    # 1. Inputs
                    in_seq = bboxes[i : i+OBS_LEN]
                    
                    # 2. Future (Ground Truth for Trajectory)
                    fut_seq = bboxes[i+OBS_LEN : i+total_len]
                    
                    # 3. Attributes (We take the attribute of the LAST observed frame as the state)
                    # "Is this sample occluded?" -> Check frame 15
                    curr_attr = attrs[i+OBS_LEN-1] 
                    
                    # 4. Intention Label (Simple Heuristic: Displacement)
                    # If they move significant X distance in future -> Crossing
                    start_x = in_seq[-1][0]
                    end_x = fut_seq[-1][0]
                    displacement = abs(end_x - start_x)
                    label_cls = 1 if displacement > 0.05 else 0 # Threshold 0.05 normalized width
                    
                    # Feature Construction [x, y, w, h]
                    feats = []
                    for b in in_seq:
                        feats.append([b[0], b[1], b[2]-b[0], b[3]-b[1]])
                    
                    # Targets Construction (Only x, y center for trajectory)
                    targets_traj = []
                    for b in fut_seq:
                        cx = (b[0] + b[2]) / 2
                        cy = (b[1] + b[3]) / 2
                        targets_traj.append([cx, cy])

                    data.append({
                        'x': np.array(feats, dtype=np.float32),      # (15, 4)
                        'y_cls': label_cls,                          # (1)
                        'y_traj': np.array(targets_traj, dtype=np.float32), # (45, 2)
                        'meta': {
                            'occ': curr_attr['occ'],
                            'int': curr_attr['int'],
                            'sig': is_sig,
                            'night': is_night
                        }
                    })
    return data

if not os.path.exists(ADVANCED_DATA_PATH):
    print("Processing advanced dataset...")
    advanced_data = process_dataset_advanced(DATA_SETS)
    with open(ADVANCED_DATA_PATH, 'wb') as f: pickle.dump(advanced_data, f)
else:
    print("Loading cached advanced data...")
    with open(ADVANCED_DATA_PATH, 'rb') as f: advanced_data = pickle.load(f)

print(f"Total Samples: {len(advanced_data)}")

Processing advanced dataset...


Set gp_set_0009: 100%|██████████████████████████████████████████████████████████████| 1/1 [00:00<00:00,  1.07it/s]


Total Samples: 153043


In [15]:
class AdvancedPedDataset(Dataset):
    def __init__(self, data):
        self.data = data
        
    def __len__(self):
        return len(self.data)
    
    def __getitem__(self, idx):
        item = self.data[idx]
        x = torch.tensor(item['x'], dtype=torch.float32)
        y_cls = torch.tensor(item['y_cls'], dtype=torch.float32).unsqueeze(0)
        y_traj = torch.tensor(item['y_traj'], dtype=torch.float32) # Shape (45, 2)
        
        # We need to return metadata as simple tensors for batching
        meta = torch.tensor([
            item['meta']['occ'],
            item['meta']['sig'],
            item['meta']['night'],
            item['meta']['int']
        ], dtype=torch.int8)
        
        return x, y_cls, y_traj, meta

train_data, test_data = train_test_split(advanced_data, test_size=0.3, random_state=42)
train_loader = DataLoader(AdvancedPedDataset(train_data), batch_size=64, shuffle=True)
test_loader = DataLoader(AdvancedPedDataset(test_data), batch_size=1, shuffle=False) # BS=1 for easy eval

In [16]:
class IDDPedModel(nn.Module):
    def __init__(self):
        super(IDDPedModel, self).__init__()
        
        # Shared Encoder
        self.encoder = nn.LSTM(4, 128, batch_first=True)
        
        # Task 1: Intention (PIP) Head
        self.pip_head = nn.Sequential(
            nn.Linear(128, 64),
            nn.ReLU(),
            nn.Linear(64, 1),
            nn.Sigmoid()
        )
        
        # Task 2: Trajectory (PTP) Decoder
        self.ptp_decoder = nn.LSTM(2, 128, batch_first=True) # Input is prev coord
        self.ptp_out = nn.Linear(128, 2) # Predict x, y
        
    def forward(self, x, future_len=45):
        # x: (Batch, 15, 4)
        _, (h, c) = self.encoder(x)
        
        # --- PIP Output ---
        pip_pred = self.pip_head(h[-1])
        
        # --- PTP Output (Autoregressive decoding) ---
        # Initialize decoder input with the last observed position (center x, y)
        last_box = x[:, -1, :] 
        curr_pos = torch.stack([
            (last_box[:, 0] + last_box[:, 2])/2, 
            (last_box[:, 1] + last_box[:, 3])/2
        ], dim=1).unsqueeze(1) # (Batch, 1, 2)
        
        ptp_preds = []
        decoder_h, decoder_c = h, c # Init decoder state with encoder state
        
        for _ in range(future_len):
            out, (decoder_h, decoder_c) = self.ptp_decoder(curr_pos, (decoder_h, decoder_c))
            xy_pred = self.ptp_out(out) # (Batch, 1, 2)
            ptp_preds.append(xy_pred)
            curr_pos = xy_pred # Feed prediction as next input
            
        ptp_pred = torch.cat(ptp_preds, dim=1) # (Batch, 45, 2)
        
        return pip_pred, ptp_pred

model = IDDPedModel().to(device)
opt = optim.Adam(model.parameters(), lr=1e-3)
crit_cls = nn.BCELoss()
crit_reg = nn.MSELoss()

In [17]:
def train_multitask(epochs=10):
    model.train()
    for ep in range(epochs):
        loop = tqdm(train_loader, desc=f"Epoch {ep+1}", leave=False)
        epoch_loss = 0
        
        for x, y_c, y_t, _ in loop:
            x, y_c, y_t = x.to(device), y_c.to(device), y_t.to(device)
            
            opt.zero_grad()
            p_c, p_t = model(x)
            
            # Loss = Classification Loss + Regression Loss
            l_cls = crit_cls(p_c, y_c)
            l_reg = crit_reg(p_t, y_t)
            loss = l_cls + (l_reg * 10) # Weight regression higher usually
            
            loss.backward()
            opt.step()
            epoch_loss += loss.item()
            
        print(f"Epoch {ep+1} Loss: {epoch_loss/len(train_loader):.4f}")

train_multitask(epochs=5)
# Save
torch.save(model.state_dict(), os.path.join(OUTPUT_ROOT, "idd_multitask_model.pth"))

Epoch 1 Loss: 0.7044


Epoch 2 Loss: 0.5728


Epoch 3 Loss: 0.5287


Epoch 4 Loss: 0.4912


Epoch 5 Loss: 0.4648


In [18]:
def generate_tables(loader, model):
    model.eval()
    
    # Storage for categorization
    # Structure: 'condition_name': {'true_labels': [], 'pred_scores': [], 'mse_vals': []}
    categories = {
        'all': {'y':[], 'p':[], 'mse':[]},
        'occ_no': {'y':[], 'p':[], 'mse':[]}, 'occ_yes': {'y':[], 'p':[], 'mse':[]},
        'sig_yes': {'y':[], 'p':[], 'mse':[]}, 'sig_no': {'y':[], 'p':[], 'mse':[]},
        'day': {'y':[], 'p':[], 'mse':[]}, 'night': {'y':[], 'p':[], 'mse':[]},
        'int_no': {'y':[], 'p':[], 'mse':[]}, 'int_yes': {'y':[], 'p':[], 'mse':[]}
    }
    
    print("Evaluating...")
    with torch.no_grad():
        for x, y_c, y_t, meta in loader:
            x, y_c, y_t = x.to(device), y_c.to(device), y_t.to(device)
            p_c, p_t = model(x)
            
            # Metadata: [occ, sig, night, int]
            occ = meta[0, 0].item()
            sig = meta[0, 1].item()
            night = meta[0, 2].item()
            inter = meta[0, 3].item()
            
            # Metrics
            # PIP: Scores
            score = p_c.item()
            label = y_c.item()
            
            # PTP: MSE at 1.5s (Last frame = index 44)
            # Coordinates are normalized (0-1). Paper reports pixels.
            # Assuming 1920x1080 resolution for conversion back to pixels
            pred_end = p_t[0, -1, :].cpu().numpy()
            true_end = y_t[0, -1, :].cpu().numpy()
            
            # Scale to pixels (approx average dimension)
            scale = np.array([1920, 1080])
            mse_val = np.mean(((pred_end - true_end) * scale)**2)
            
            # --- Bucketing ---
            def add_to(key, l, s, m):
                categories[key]['y'].append(l)
                categories[key]['p'].append(s)
                categories[key]['mse'].append(m)

            add_to('all', label, score, mse_val)
            
            add_to('occ_yes' if occ else 'occ_no', label, score, mse_val)
            add_to('sig_yes' if sig else 'sig_no', label, score, mse_val)
            add_to('night' if night else 'day', label, score, mse_val)
            add_to('int_yes' if inter else 'int_no', label, score, mse_val)

    # --- Print Table IV (AUC) ---
    print("\n" + "="*50)
    print("TABLE IV: PIP Evaluation (AUC Scores)")
    print("="*50)
    print(f"{'Condition':<15} | {'Variant':<10} | {'AUC':<10}")
    print("-" * 40)
    
    for cat, data in categories.items():
        if len(data['y']) < 10 or len(set(data['y'])) < 2: 
            auc = "N/A"
        else:
            auc = roc_auc_score(data['y'], data['p'])
            auc = f"{auc:.3f}"
        print(f"{cat:<15} | {'-':<10} | {auc:<10}")

    # --- Print Table V (MSE) ---
    print("\n" + "="*50)
    print("TABLE V: PTP Evaluation (MSE @ 1.5s)")
    print("="*50)
    print(f"{'Condition':<15} | {'Variant':<10} | {'MSE':<10}")
    print("-" * 40)
    
    for cat, data in categories.items():
        if len(data['mse']) == 0:
            mse = "N/A"
        else:
            mse = np.mean(data['mse'])
            mse = f"{mse:.0f}"
        print(f"{cat:<15} | {'-':<10} | {mse:<10}")

# Run
generate_tables(test_loader, model)

Evaluating...

TABLE IV: PIP Evaluation (AUC Scores)
Condition       | Variant    | AUC       
----------------------------------------
all             | -          | 0.864     
occ_no          | -          | 0.869     
occ_yes         | -          | 0.835     
sig_yes         | -          | 0.862     
sig_no          | -          | 0.897     
day             | -          | 0.860     
night           | -          | 0.899     
int_no          | -          | 0.864     
int_yes         | -          | N/A       

TABLE V: PTP Evaluation (MSE @ 1.5s)
Condition       | Variant    | MSE       
----------------------------------------
all             | -          | 13832     
occ_no          | -          | 14571     
occ_yes         | -          | 11042     
sig_yes         | -          | 13772     
sig_no          | -          | 14817     
day             | -          | 13800     
night           | -          | 14180     
int_no          | -          | 13832     
int_yes         | -          